In [1]:
import os

# Prepend the folder containing cdo to PATH
os.environ["PATH"] = "/sw/spack-levante/cdo-2.2.2-4z4icb/bin:" + os.environ["PATH"]

from cdo import Cdo
cdo = Cdo()
print(cdo.version())

2.2.2


In [2]:
import glob
import xarray as xr
import numpy as np
from scipy.special import lambertw
import metpy.calc as mpcalc
from metpy.units import units


# Paths
base_path    = "/work/uc1275/u301827/02_MSE/full_midlatitude/raw"
scratch_path = "/scratch/u/u301827/full_midlatitude/"
out_file     = "/work/uc1275/u301827/02_MSE/full_midlatitude/era5_midlatitudes_JJA_compiled.nc"

In [3]:
def cdo_split_monthly_py(in_file, scratch_dir, prefix="era5_midlat_JJA_"):
    """
    Split a NetCDF file into monthly files using python-cdo.

    Output files:
      scratch_dir/prefixYYYYMM.nc
    """
    cdo = Cdo()
    os.makedirs(scratch_dir, exist_ok=True)
    out_prefix = os.path.join(scratch_dir, prefix)

    # This creates multiple files automatically
    cdo.splityearmon(
        input=in_file,
        output=out_prefix,
        options="-O"
    )

    return out_prefix

IN_DS   = "/work/uc1275/u301827/02_MSE/full_midlatitude/era5_midlatitudes_JJA_compiled.nc"
ZS_PATH = "/work/uc1275/u301827/02_MSE/surface_geopotential_zs_fullres.nc"

OUT_ALL = "/work/uc1275/u301827/02_MSE/full_midlatitude/era5_midlatitudes_JJA_compiled_clean_plusDerived.nc"
IN_DS = "/work/uc1275/u301827/02_MSE/full_midlatitude/era5_midlatitudes_JJA_compiled.nc"

scratch_path = "/scratch/u/u301827/full_midlatitude/"

cdo_split_monthly_py(IN_DS, scratch_path, prefix="era5_midlat_JJA_")

import glob
monthly_files = sorted(glob.glob(os.path.join(scratch_path, "era5_midlat_JJA_*.nc")))
print(len(monthly_files), monthly_files[:3])



258 ['/scratch/u/u301827/full_midlatitude/era5_midlat_JJA_194006.nc', '/scratch/u/u301827/full_midlatitude/era5_midlat_JJA_194007.nc', '/scratch/u/u301827/full_midlatitude/era5_midlat_JJA_194008.nc']


In [4]:
import numpy as np
import xarray as xr
from scipy.special import lambertw
import metpy.calc as mpcalc
from metpy.units import units


# -----------------------
# Helpers
# -----------------------
def _align_metpy_to_ref(metpy_out, ref_da):
    """
    Take MetPy output (often a DataArray or pint Quantity) and return a NumPy array
    aligned to the ref_da dimension order.

    ref_da is the variable you want to match (e.g., ds["tasmax"]).
    """
    # Case 1: MetPy returned an xarray DataArray (this is your case)
    if isinstance(metpy_out, xr.DataArray):
        # Reorder dims to match reference
        out = metpy_out.transpose(*ref_da.dims)
        # Strip units if present
        if hasattr(out.data, "magnitude"):
            return np.asarray(out.data.magnitude)
        return np.asarray(out.data)

    # Case 2: MetPy returned a pint Quantity
    if hasattr(metpy_out, "magnitude"):
        arr = np.asarray(metpy_out.magnitude)
    else:
        arr = np.asarray(metpy_out)

    # If it's already same shape as ref, assume same order
    if arr.shape == ref_da.shape:
        return arr

    # Otherwise, last-resort: try to reshape by matching sizes
    # (works when it's just permuted)
    ref_shape = ref_da.shape
    if sorted(arr.shape) == sorted(ref_shape):
        # Build permutation from arr axes to ref axes by matching lengths
        perm = []
        used = set()
        for n in ref_shape:
            for ax, m in enumerate(arr.shape):
                if ax in used:
                    continue
                if m == n:
                    perm.append(ax)
                    used.add(ax)
                    break
        return np.transpose(arr, axes=perm)

    raise ValueError(f"Cannot align MetPy output shape {arr.shape} to reference shape {ref_da.shape}")


def _squeeze_singletons(ds, candidates=("depth", "depth_2", "lev", "nhyi", "nhym", "time")):
    """Squeeze only singleton dims that exist."""
    squeeze_dims = [d for d in candidates if d in ds.dims and ds.sizes[d] == 1]
    return ds.squeeze(squeeze_dims, drop=True) if squeeze_dims else ds


def attach_surface_geopotential(ds, zs_path, z_var_in_file="z", out_var="zs", align_method="nearest"):
    """
    Monthly-safe, eager attach of invariant surface geopotential (ERA5 param 129).

    No chunking, no persist, no parallelization.
    """
    if ("lat" not in ds.coords) or ("lon" not in ds.coords):
        raise KeyError("ds must have 'lat' and 'lon' coordinates.")

    zs_ds = xr.open_dataset(zs_path)
    zs_ds = _squeeze_singletons(zs_ds, candidates=("time",))

    if z_var_in_file not in zs_ds.data_vars:
        raise KeyError(f"Expected '{z_var_in_file}' in {zs_path}, found: {list(zs_ds.data_vars)}")

    zs = zs_ds[z_var_in_file].rename(out_var)

    # align to monthly ds grid
    zs = zs.interp(lat=ds["lat"], lon=ds["lon"], method=align_method)

    zs.attrs.update({
        "long_name": "Surface geopotential",
        "units": "m^2 s^-2",
        "source": "ERA5 invariant surface geopotential (param 129)",
    })

    return ds.assign({out_var: zs})


# -----------------------
# MetPy RH from dewpoint
# -----------------------
def compute_rh_from_T_Td_metpy(ds, T_var="tasmax", Td_var="2d", out_name="rh"):
    """
    RH from temperature and dewpoint using MetPy.
    Inputs expected in K.
    Returns RH in 0..1.
    """
    T = ds[T_var] * units.kelvin
    Td = ds[Td_var] * units.kelvin
    rh = mpcalc.relative_humidity_from_dewpoint(T, Td)  # unitless (dimensionless)
    rh_da = xr.DataArray(rh, coords=ds[T_var].coords, dims=ds[T_var].dims, name=out_name)
    rh_da.attrs.update({"long_name": "Relative humidity", "units": "1"})
    return rh_da


# -----------------------
# LCL (LambertW, eager)
# -----------------------
def compute_LCL_metpy_inputs(T, qv, Rh, z, ps, g=9.81):
    """
    Same formulation you posted, but:
    - works with xarray DataArrays
    - eager (no dask apply_ufunc)
    - safe clipping for Rh and LambertW argument
    """
    # physical constants
    Ttr = 273.16
    E0v = 2.3740e6
    cvl = 4119
    cvv = 1418
    Rv  = 461
    cpv = cvv + Rv
    Ra  = 287.04
    cva = 719
    cpa = cva + Ra

    # numpy arrays (eager)
    Tn  = np.asarray(T)
    qvn = np.asarray(qv)
    Rhn = np.asarray(Rh)
    zn  = np.asarray(z)
    psn = np.asarray(ps)

    Rm  = (1 - qvn) * Ra + qvn * Rv
    cpm = (1 - qvn) * cpa + qvn * cpv

    a = cpm / Rm + (cvl - cpv) / Rv
    b = - (E0v - Ttr * (cvv - cvl)) / (Rv * Tn)
    c = b / a

    # RH safety
    Rhn = np.clip(Rhn, 1e-6, 1.0)

    RHcec = c * np.exp(c) * (Rhn ** (1 / a))

    # clip to real domain of W_{-1}: [-1/e, 0)
    RHcec = np.clip(RHcec, -1/np.e + 1e-12, -1e-12)

    W = lambertw(RHcec, k=-1).real
    TLCL = Tn * c / W

    # z: geopotential -> height
    z_m = zn / g
    zLCL = z_m + (cpm / g) * (Tn - TLCL)

    pLCL = psn * (TLCL / Tn) ** (cpm / Rm)

    # back to DataArrays with same coords/dims as T
    TLCL_da = xr.DataArray(TLCL, coords=T.coords, dims=T.dims, name="TLCL")
    zLCL_da = xr.DataArray(zLCL, coords=T.coords, dims=T.dims, name="zLCL")
    pLCL_da = xr.DataArray(pLCL, coords=T.coords, dims=T.dims, name="pLCL")

    TLCL_da.attrs.update({"long_name": "Temperature at LCL", "units": "K"})
    zLCL_da.attrs.update({"long_name": "Height of LCL (from surface reference)", "units": "m"})
    pLCL_da.attrs.update({"long_name": "Pressure at LCL", "units": "Pa"})

    return TLCL_da, zLCL_da, pLCL_da


# -----------------------
# MetPy MSEs (monthly-safe)
# -----------------------
def compute_surface_mse_metpy(ds, T_var="tasmax", q_var="q", zs_var="zs", out_name="mse"):
    if zs_var not in ds:
        raise KeyError(f"'{zs_var}' missing. Attach it first (surface geopotential).")

    Tref = ds[T_var]  # reference for dims/coords
    T = Tref * units.kelvin
    q = ds[q_var] * units.dimensionless
    z = mpcalc.geopotential_to_height(ds[zs_var] * units("m^2/s^2"))

    mse = mpcalc.moist_static_energy(z, T, q)

    mse_vals = _align_metpy_to_ref(mse, Tref)
    mse_da = xr.DataArray(mse_vals, coords=Tref.coords, dims=Tref.dims, name=out_name)
    mse_da.attrs.update({"long_name": "Surface moist static energy", "units": "J kg-1"})
    return mse_da.to_dataset()



def compute_sat_mse_metpy(ds, hPa=500, T_var="t", z_var="z", out_name="mse_sat"):
    Tref = ds[T_var]
    T = Tref * units.kelvin
    z = mpcalc.geopotential_to_height(ds[z_var] * units("m^2/s^2"))

    p_sat = mpcalc.saturation_vapor_pressure(T)
    q_sat = 0.622 * p_sat / (hPa * units.hPa)

    mse_sat = mpcalc.moist_static_energy(z, T, q_sat)

    mse_vals = _align_metpy_to_ref(mse_sat, Tref)
    mse_da = xr.DataArray(mse_vals, coords=Tref.coords, dims=Tref.dims, name=out_name)
    mse_da.attrs.update({"long_name": f"Saturated moist static energy at {hPa} hPa", "units": "J kg-1"})
    return mse_da.to_dataset()




# -----------------------
# Temperature bounds (as you had it)
# -----------------------
def compute_temperature_bounds(
    ds,
    T500_var="t",
    ps_var="sp",
    mse_var="mse",
    mse_sat_var="mse_sat",
    p_ref=500e2,
    R=287.0,
    cp=1004.0,
):
    ps = ds[ps_var]
    T500 = ds[T500_var]
    ds = ds.copy()
    ds["t_bound"] = T500 * (ps / p_ref) ** (R / cp)
    ds["t_bound_mse"] = (-ds[mse_var] + ds[mse_sat_var]) * 1e3 / cp
    return ds


# -----------------------
# Full monthly pipeline (eager, MetPy)
# -----------------------
def compute_all_monthly_metpy(
    ds_month,
    zs_path,
    use_at_tasmax=True,
    hPa=500,
    zs_var="zs",
):
    """
    Process ONE monthly dataset (eager, single-process):
      - attach zs
      - compute mse, mse_sat
      - compute bounds
      - compute RH from dewpoint (MetPy)
      - compute LCL via LambertW
    """
    ds = _squeeze_singletons(ds_month)

    # attach invariant zs if not present
    if zs_var not in ds:
        ds = attach_surface_geopotential(ds, zs_path=zs_path, out_var=zs_var)

    # select variable names
    if use_at_tasmax:
        q_var  = "q_at_tasmax"  if "q_at_tasmax"  in ds else "q"
        Td_var = "2d_at_tasmax" if "2d_at_tasmax" in ds else "2d"
        t_var  = "t_at_tasmax"  if "t_at_tasmax"  in ds else "t"
        z_var  = "z_at_tasmax"  if "z_at_tasmax"  in ds else "z"
        sp_var = "sp_at_tasmax" if "sp_at_tasmax" in ds else "sp"
    else:
        q_var, Td_var, t_var, z_var, sp_var = "q", "2d", "t", "z", "sp"

    # MSEs
    mse_ds     = compute_surface_mse_metpy(ds, T_var="tasmax", q_var=q_var, zs_var=zs_var, out_name="mse")
    mse_sat_ds = compute_sat_mse_metpy(ds, hPa=hPa, T_var=t_var, z_var=z_var, out_name="mse_sat")
    ds = xr.merge([ds, mse_ds, mse_sat_ds], compat="override")

    # bounds
    ds = compute_temperature_bounds(ds, T500_var=t_var, ps_var=sp_var, mse_var="mse", mse_sat_var="mse_sat")

    # RH + LCL
    rh = compute_rh_from_T_Td_metpy(ds, T_var="tasmax", Td_var=Td_var, out_name="rh")
    TLCL, zLCL, pLCL = compute_LCL_metpy_inputs(ds["tasmax"], ds[q_var], rh, ds[zs_var], ds[sp_var])

    ds = ds.assign(TLCL=TLCL, zLCL=zLCL, pLCL=pLCL, rh=rh)

    return ds


In [6]:
import xarray as xr

test_month = monthly_files[30]
print("Testing:", test_month)

ds_month = xr.open_dataset(test_month)

ds_out = compute_all_monthly_metpy(
    ds_month,
    zs_path=ZS_PATH,
    use_at_tasmax=True,
    hPa=500,
)

print(ds_out)


Testing: /scratch/u/u301827/full_midlatitude/era5_midlat_JJA_195006.nc
<xarray.Dataset>
Dimensions:          (bnds: 2, lat: 89, lon: 1280, nhyi: 138, nhym: 137, time: 30)
Coordinates:
  * time             (time) datetime64[ns] 1950-06-01 1950-06-02 ... 1950-06-30
  * lon              (lon) float64 0.0 0.2812 0.5625 ... 359.2 359.4 359.7
  * lat              (lat) float64 64.78 64.5 64.22 63.93 ... 40.61 40.33 40.05
    plev             float64 5e+04
Dimensions without coordinates: bnds, nhyi, nhym
Data variables: (12/32)
    hyai             (nhyi) float64 ...
    hybi             (nhyi) float64 ...
    hyam             (nhym) float64 ...
    hybm             (nhym) float64 ...
    depth_bnds       (bnds) float64 ...
    tasmax           (time, lat, lon) float32 282.1 282.1 282.1 ... 304.9 304.6
    ...               ...
    t_bound          (time, lat, lon) float32 309.6 309.6 309.7 ... 314.1 318.0
    t_bound_mse      (time, lat, lon) float32 13.12 13.12 13.13 ... 1.531 1.866
    TLC

In [5]:
import os
import glob
import xarray as xr


def _month_tag_from_filename(path):
    base = os.path.basename(path).replace(".nc", "")
    parts = base.split("_")
    return parts[-1] if parts else base

def _carry_through_vars(ds_in, ds_out, names):
    """If a variable exists in ds_in, attach it to ds_out unchanged."""
    for name in names:
        if name in ds_in:
            ds_out[name] = ds_in[name]
    return ds_out


def process_one_month_file_serial(
    month_file,
    zs_path,
    out_dir,
    use_at_tasmax=True,
    hPa=500,
    overwrite=False,
):
    tag = _month_tag_from_filename(month_file)
    out_file = os.path.join(out_dir, f"derived_{tag}.nc")

    if (not overwrite) and os.path.exists(out_file):
        print(f"[SKIP] {tag}")
        return f"skipped {tag}"

    print(f"[RUN ] {tag}")

    ds_month = xr.open_dataset(month_file)

    ds_out = compute_all_monthly_metpy(
        ds_month,
        zs_path=zs_path,
        use_at_tasmax=use_at_tasmax,
        hPa=hPa,
    )

    # ✅ NEW: carry through t_dailymax (hourly-max t) into derived output
    ds_out = _carry_through_vars(ds_month, ds_out, ["t_dailymax", "t_dailymax_hour"])

    # Optional: reduce file size
    for v in ["mse", "mse_sat", "t_bound", "t_bound_mse", "TLCL", "zLCL", "pLCL", "rh",
              "t_dailymax", "t_dailymax_hour"]:  # <- you can also downcast these
        if v in ds_out:
            ds_out[v] = ds_out[v].astype("float32")

    os.makedirs(out_dir, exist_ok=True)
    ds_out.to_netcdf(out_file)

    ds_month.close()
    ds_out.close()

    print(f"[DONE] {tag}")
    return f"done {tag}"


def process_all_months_serial(
    scratch_dir,
    out_dir,
    zs_path,
    pattern="era5_midlat_JJA_*.nc",
    use_at_tasmax=True,
    hPa=500,
    overwrite=False,
):
    os.makedirs(out_dir, exist_ok=True)

    monthly_files = sorted(glob.glob(os.path.join(scratch_dir, pattern)))
    if not monthly_files:
        raise FileNotFoundError(f"No files found: {os.path.join(scratch_dir, pattern)}")

    results = []

    print(f"[INFO] Processing {len(monthly_files)} monthly files (serial)")

    for f in monthly_files:
        res = process_one_month_file_serial(
            f,
            zs_path=zs_path,
            out_dir=out_dir,
            use_at_tasmax=use_at_tasmax,
            hPa=hPa,
            overwrite=overwrite,
        )
        results.append(res)

    done = sum(r.startswith("done") for r in results)
    skipped = sum(r.startswith("skipped") for r in results)
    print(f"[SUMMARY] done={done}, skipped={skipped}, total={len(results)}")

    return results


In [6]:
SCRATCH_DIR = "/scratch/u/u301827/full_midlatitude/"
OUT_DIR = "/work/uc1275/u301827/02_MSE/full_midlatitude/derived_monthly_metpy"
ZS_PATH = "/work/uc1275/u301827/02_MSE/surface_geopotential_zs_fullres.nc"

results = process_all_months_serial(
    scratch_dir=SCRATCH_DIR,
    out_dir=OUT_DIR,
    zs_path=ZS_PATH,
    overwrite=False
)


[INFO] Processing 258 monthly files (serial)
[SKIP] 194006
[SKIP] 194007
[SKIP] 194008
[SKIP] 194106
[SKIP] 194107
[SKIP] 194108
[SKIP] 194206
[SKIP] 194207
[SKIP] 194208
[SKIP] 194306
[SKIP] 194307
[SKIP] 194308
[SKIP] 194406
[SKIP] 194407
[SKIP] 194408
[SKIP] 194506
[SKIP] 194507
[SKIP] 194508
[SKIP] 194606
[SKIP] 194607
[SKIP] 194608
[SKIP] 194706
[SKIP] 194707
[SKIP] 194708
[SKIP] 194806
[SKIP] 194807
[SKIP] 194808
[SKIP] 194906
[SKIP] 194907
[SKIP] 194908
[SKIP] 195006
[SKIP] 195007
[SKIP] 195008
[SKIP] 195106
[SKIP] 195107
[SKIP] 195108
[SKIP] 195206
[SKIP] 195207
[SKIP] 195208
[SKIP] 195306
[SKIP] 195307
[SKIP] 195308
[SKIP] 195406
[SKIP] 195407
[SKIP] 195408
[SKIP] 195506
[SKIP] 195507
[SKIP] 195508
[SKIP] 195606
[SKIP] 195607
[SKIP] 195608
[SKIP] 195706
[SKIP] 195707
[SKIP] 195708
[SKIP] 195806
[SKIP] 195807
[SKIP] 195808
[SKIP] 195906
[SKIP] 195907
[SKIP] 195908
[SKIP] 196006
[SKIP] 196007
[SKIP] 196008
[SKIP] 196106
[SKIP] 196107
[SKIP] 196108
[SKIP] 196206
[SKIP] 196207
[SK